# 13 · When the model itself is wrong

Chapter 08 taught us to compute uncertainty and resolution. But those numbers
carry a hidden fine print: they are correct only *if the model is right*. They
assume the fault geometry, the elastic Earth, the mesh, and the noise
description are all exactly as declared. When one of those assumptions is wrong,
the inversion can be confidently, precisely wrong, and the formal error bars
will not warn you. This chapter shows how to probe for that kind of error, which
is the difference between a precise answer and an accurate one.

**Learning objectives**

By the end of the chapter, you will be able to:

- distinguish conditional precision from accuracy under a wrong model
- design a clean one-factor sensitivity experiment
- spot the coherent spatial residuals that betray model error
- report bias alongside formal uncertainty

**Prerequisites:** Chapters 08 and 09
**Estimated time:** 45–60 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Precision is not accuracy

The posterior covariance and resolution of Chapter 08 are computed *given* a
particular $G$ and noise covariance $C_d$. They faithfully describe how much the
estimate would wobble under fresh noise, and how the inversion averages the
truth, *within that assumed model*. What they cannot see is that the assumed
model might be the wrong one.

The distinction is precision versus accuracy. **Precision** is repeatability: a
tight error bar means fresh data would give nearly the same answer.
**Accuracy** is closeness to the truth. A dartboard makes the difference
concrete: a tight cluster of darts far from the bullseye is precise but
inaccurate. A wrong fault geometry produces exactly that, a tight, wrong answer,
and the formal error bar reports only the tightness.

## 2. Where the bias comes from

Suppose the real Earth produces data through one operator, call it $G_t$, but we
invert using a different one, $G_a$ (for example, the wrong dip). Even with
perfectly clean, zero-mean noise, the expected estimate is not the truth; it is
shifted by a fixed amount that depends on the mismatch between $G_a$ and $G_t$.

This shift is a **bias**, and it is entirely absent from the covariance computed
with $G_a$, because that calculation never suspects $G_a$ is wrong. Sometimes
the wrong model simply cannot reproduce the data and the residuals give it away;
but often the slip quietly distorts to absorb the mismatch, keeping the fit
respectable while the slip map lies. The only way to find such error is to
*change the assumption and see what moves*.

## 3. An experiment: the right data, the wrong dip

We run a controlled test. Generate data from a fault with a true dip of 25
degrees, then invert those same data twice: once with the correct 25-degree
fault, and once with a wrong 30-degree fault, a modest 5-degree error of the
kind you might easily make in practice. Everything else, the stations, the
regularization, the noise realization, is held identical, so any difference is
due to the dip alone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# Shared geometry; only the dip differs between truth and the wrong model.
geometry = dict(
    lat=34.0, lon=-118.0, depth=10_000.0, strike=90.0,
    length=36_000.0, width=18_000.0, n_length=6, n_width=3,
)
truth_fault = geodef.Fault.planar(dip=25.0, **geometry)
wrong_fault = geodef.Fault.planar(dip=30.0, **geometry)

In [ ]:
# A compact patch of true dip slip on the correct 25-degree fault.
along = np.arange(truth_fault.n_patches) % 6
down = np.arange(truth_fault.n_patches) // 6
true_slip = 1.2 * np.exp(-((along - 2.5) / 1.5) ** 2 - ((down - 1.0) / 0.9) ** 2)

In [ ]:
# Forward-model the truth to GNSS and add 3 mm noise.
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.25, -117.75, 8), np.linspace(33.83, 34.18, 6)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = truth_fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.003
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="synthetic_gnss",
)

Invert with each fault, using identical settings, and compare the fit.

In [ ]:
solve_kwargs = dict(
    datasets=gnss, components="dip", regularization="laplacian",
    regularization_strength=1.0, bounds=(0.0, None),
)
right = geodef.solve(truth_fault, **solve_kwargs)
wrong = geodef.solve(wrong_fault, **solve_kwargs)
print(f"reduced chi-squared: right dip {right.reduced_chi2:.2f}, "
      f"wrong dip {wrong.reduced_chi2:.2f}")

The wrong dip pushes the reduced chi-squared up (from about 1 to a few), which
is a genuine clue that something is off. But it is not a dramatic blow-up: a
hurried analyst might shrug at a chi-squared of a few and move on. The slip does
its best to absorb the geometry error, keeping the fit merely mediocre rather
than obviously broken.

## 4. Bias measured against the formal error bar

The telling comparison is the size of the bias (recovered minus true slip on the
wrong fault) relative to the formal uncertainty that inversion reports. If bias
is many times the error bar, the formal uncertainty is badly understating how
wrong the answer is.

In [ ]:
formal_sigma = geodef.invert.model_uncertainty(wrong, wrong_fault, gnss)
bias = wrong.dip_slip - true_slip
print(f"largest |bias| / formal sigma: {np.max(np.abs(bias) / formal_sigma):.1f}")

The largest bias is several times the formal error bar. In other words, the
inversion is confidently wrong: it reports a tight uncertainty around a slip
value that is off by far more than that uncertainty admits. Look at the three
slip maps together, the truth, the correct-dip recovery, and the wrong-dip
recovery.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
limit = max(true_slip.max(), right.dip_slip.max(), wrong.dip_slip.max())
panels = [
    (axes[0], truth_fault, true_slip, "Truth: dip 25"),
    (axes[1], truth_fault, right.dip_slip, "Recovered: dip 25"),
    (axes[2], wrong_fault, wrong.dip_slip, "Recovered: assumed dip 30"),
]
for ax, patch_fault, field, title in panels:
    geodef.plot.slip(patch_fault, field, ax=ax, vmin=0, vmax=limit,
                     title=title, colorbar_label="Slip (m)")
plt.show()

## 5. Residual forensics

The chi-squared told us *that* the fit worsened, but not *why* or *where*. The
spatial pattern of the residuals answers both. When the geometry is wrong, the
leftover misfit is not random scatter; it forms coherent, organized structure,
because the wrong model fails in the same systematic way across neighboring
stations. Here is the vertical residual of the wrong-dip inversion, mapped by
station.

In [ ]:
residual_up = wrong.residuals[2::3]      # vertical component of each station residual
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
points = ax.scatter(station_lon, station_lat, c=residual_up * 1e3, cmap="RdBu_r")
ax.set(xlabel="longitude (degrees)", ylabel="latitude (degrees)",
       title="Vertical residual, wrong-dip inversion")
fig.colorbar(points, ax=ax, label="residual (mm)")
plt.show()

The residual has coherent spatial structure: a systematic, mostly one-signed
pattern concentrated near the fault, rather than the random speckle that pure
noise would produce. That organized shape is the fingerprint of a wrong forward
model; noise alone would look uncorrelated from station to station. A scalar
RMS averages this signal away, which is why you should always look at the
spatial and per-component residual maps, not just one number.

## 6. Turning this into a habit: sensitivity analysis

The experiment above is a template. To probe any assumption, change *one*
declared thing (the dip, Poisson's ratio, the patch size, the noise
covariance), hold everything else fixed including the noise realization, and
tabulate how a quantity you care about (peak slip, moment magnitude, coupling)
responds. Doing this across the plausible range of each assumption produces a
**sensitivity table** that belongs right next to the formal error bars, because
it measures exactly what those error bars omit.

**Checkpoint.** The wrong dip raised the reduced chi-squared only modestly, not
to an obviously broken level, yet the slip was biased several times its formal
error bar. How does the fit stay merely mediocre while the slip goes so wrong?

<details><summary>Answer</summary>

The slip distribution has enough freedom to partly compensate for the wrong
geometry: it rearranges itself so the predicted surface data still roughly match
the observations, holding the misfit down. The compensation is imperfect, so the
chi-squared does creep up, but not enough to look alarming, while the slip needed
to achieve it is badly distorted. This is why a not-terrible fit is not proof of
a correct model, and why sensitivity analysis is necessary.

</details>

## Checkpoint questions

**Why is the formal uncertainty unchanged by our uncertainty about the dip?**

<details><summary>Answer</summary>

The covariance is computed conditional on the chosen dip; in that calculation
the dip is a fixed constant, not a random variable. So doubt about the dip
simply does not enter the formal error bar. It has to be probed separately, by
changing the dip and re-inverting.

</details>

**Can featureless, random-looking residuals prove the model is correct?**

<details><summary>Answer</summary>

No. Model error can be absorbed by the slip or hidden beneath the noise level,
leaving residuals that look fine. Clean residuals are reassuring but not proof;
a wrong model can still pass this check.

</details>

**What makes a sensitivity experiment interpretable?**

<details><summary>Answer</summary>

Changing exactly one declared assumption while holding the data, the noise
realization, and every other analysis choice fixed. Then any change in the
result can be attributed cleanly to that one assumption.

</details>

## Common mistakes

- **Calling conditional error bars "total uncertainty."** They omit model-form
  error, which this chapter shows can dominate.
- **Changing two things at once.** Varying geometry and regularization together
  makes the resulting bias impossible to attribute.
- **Judging fit by scalar RMS alone.** The spatial and per-component residual
  structure carries information a single number discards.
- **Cherry-picking sensitivity cases.** Running only the tests that leave your
  preferred interpretation intact defeats the purpose.

## Recap

- Formal covariance quantifies uncertainty only inside the assumed model
  family; it is precision, not accuracy.
- A wrong geometry or Earth model can bias the estimate by many formal standard
  deviations, invisibly to the covariance.
- Coherent spatial residuals are the fingerprint of model error; scalar fit
  statistics hide them.
- One-factor sensitivity tables report the bias that formal uncertainty omits.

Chapter 14 puts uncertainty on a fully probabilistic footing with Bayesian
sampling, while carrying this chapter's warning forward: even a well-sampled
posterior is conditional on the model being right.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/13_model_misspecification_solutions.ipynb`.

1. **Dip sensitivity.** Sweep the assumed dip across a range and plot the
   resulting bias in moment magnitude.
2. **Wrong noise model.** Generate spatially correlated noise but invert with a
   diagonal covariance, and measure the effect on the estimate and its error
   bars.
3. **Mesh aliasing.** Coarsen the inversion mesh below the scale of the true
   slip and locate where aliased slip appears.
4. **Challenge: station design.** Redesign the station layout to make a 5 km
   depth error as easy as possible to detect in the residuals.

## Further reading

- Tarantola, A. (2005), *Inverse Problem Theory*, on model assumptions and
  their consequences.
- Box, G. E. P. (1976), "Science and statistics," *Journal of the American
  Statistical Association*, 71(356), 791–799, on models being useful yet wrong.
- Segall, P. (2010), *Earthquake and Volcano Deformation*, on the limits of
  elastic models in geodesy.
- [GeoDef inversion documentation](../docs/invert.md) for the residual and
  uncertainty interfaces used here.
- The next course chapter is `14_bayesian_inversion.ipynb`.